# 🛡️ ZENTRA — PPE Detection Training
### Google Colab Version (GPU T4 ฟรี)

**ขั้นตอน:**
1. Runtime → Change runtime type → **T4 GPU** ✅
2. รัน Cell ทีละ Cell ตามลำดับ
3. ใช้เวลาประมาณ **45-90 นาที** (100 epochs บน T4)
4. Download `ppe_finetuned.pt` ไปใส่ใน ZENTRA

---
⚠️ **สำคัญ**: Colab ฟรีมี session timeout ~12 ชั่วโมง  
ถ้าหลุด → รัน Cell 'Resume Training' ได้เลย

In [ ]:
# ════════════════════════════════════════════════
# CELL 1: ตรวจสอบ GPU
# ════════════════════════════════════════════════
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  ไม่พบ GPU — ไปที่ Runtime > Change runtime type > T4 GPU')

In [ ]:
# ════════════════════════════════════════════════
# CELL 2: ติดตั้ง packages
# ════════════════════════════════════════════════
!pip install -q ultralytics roboflow
print('✅ ติดตั้งเสร็จ')

In [ ]:
# ════════════════════════════════════════════════
# CELL 3: ตั้งค่า — แก้ค่าตรงนี้เท่านั้น
# ════════════════════════════════════════════════
import os

# ── ใส่ API Key ของคุณ ──────────────────────────
ROBOFLOW_API_KEY   = '8xTIheqbzg4mkSLOdFe6'   # ← เปลี่ยนถ้าต้องการ
ROBOFLOW_WORKSPACE = 'pholawats-workspace'      # ← workspace ของคุณ

# ── Training Hyperparameters ────────────────────
EPOCHS      = 100    # แนะนำ 100 (T4 ~60 min) หรือ 150 ถ้าเวลาพอ
BATCH_SIZE  = 16     # T4 16GB → batch 16 พอดี
IMG_SIZE    = 640
BASE_MODEL  = 'yolov8m.pt'  # n=เร็ว, m=สมดุล, l=แม่น (แนะนำ m)

# ── Paths ───────────────────────────────────────
BASE_DIR    = '/content/zentra'
DATA_DIR    = f'{BASE_DIR}/data'
MODELS_DIR  = f'{BASE_DIR}/models'
MERGED_DIR  = f'{DATA_DIR}/merged_ppe'

os.makedirs(DATA_DIR,   exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(MERGED_DIR, exist_ok=True)

print('✅ Config ตั้งค่าแล้ว')
print(f'   Base model : {BASE_MODEL}')
print(f'   Epochs     : {EPOCHS}')
print(f'   Batch size : {BATCH_SIZE}')

In [ ]:
# ════════════════════════════════════════════════
# CELL 4: Mount Google Drive (บันทึก model ถาวร)
# ════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

GDRIVE_DIR = '/content/drive/MyDrive/ZENTRA_Models'
os.makedirs(GDRIVE_DIR, exist_ok=True)
print(f'✅ Google Drive mounted → {GDRIVE_DIR}')
print('   Model จะถูกบันทึกที่นี่ทุกครั้ง (ไม่หายเมื่อ session หมด)')

In [ ]:
# ════════════════════════════════════════════════
# CELL 5: ดาวน์โหลด Dataset จาก Roboflow Universe
# หลาย dataset รวมกัน ~15,000 ภาพ
# ════════════════════════════════════════════════
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# รายการ dataset (เพิ่ม/ลดได้)
DATASETS = [
    # dataset หลักของทีม
    {'ws': ROBOFLOW_WORKSPACE, 'proj': 'ppe-cpxsz',                    'ver': 2, 'dest': f'{DATA_DIR}/ds_main'},
    # Public datasets จาก Roboflow Universe
    {'ws': 'roboflow-universe-projects', 'proj': 'construction-site-safety', 'ver': 1, 'dest': f'{DATA_DIR}/ds_construction'},
    {'ws': 'joseph-nelson',              'proj': 'hard-hat-workers',         'ver': 2, 'dest': f'{DATA_DIR}/ds_hardhat'},
    {'ws': 'roboflow-universe-projects', 'proj': 'ppe-detection-ljb7d',      'ver': 4, 'dest': f'{DATA_DIR}/ds_ppe2'},
]

downloaded = []
for ds in DATASETS:
    dest = ds['dest']
    import glob
    if glob.glob(f"{dest}/**/*.jpg", recursive=True):
        print(f'✅ Already exists: {ds["proj"]}')
        downloaded.append(dest)
        continue
    print(f'\n📥 Downloading: {ds["proj"]}...')
    try:
        proj = rf.workspace(ds['ws']).project(ds['proj'])
        proj.version(ds['ver']).download('yolov8', location=dest, overwrite=True)
        downloaded.append(dest)
        imgs = len(glob.glob(f"{dest}/**/*.jpg", recursive=True))
        print(f'   ✅ {imgs} images')
    except Exception as e:
        print(f'   ⚠️  ข้าม {ds["proj"]}: {e}')

print(f'\n📦 ดาวน์โหลดสำเร็จ {len(downloaded)}/{len(DATASETS)} datasets')

In [ ]:
# ════════════════════════════════════════════════
# CELL 6: Merge + Remap Classes → ZENTRA Classes
# ════════════════════════════════════════════════
import cv2
import yaml
import random
import shutil
import numpy as np
from pathlib import Path

# ZENTRA class definitions
ZENTRA_CLASSES = [
    'helmet', 'no_helmet', 'vest', 'no_vest',
    'gloves', 'no_gloves', 'goggles', 'no_goggles',
    'safety_boots', 'no_safety_boots', 'person'
]
CLS_MAP = {name: i for i, name in enumerate(ZENTRA_CLASSES)}

ALIAS_MAP = {
    'hard-hat':'helmet','hardhat':'helmet','hard_hat':'helmet','helmet':'helmet','head':'helmet',
    'no-hardhat':'no_helmet','no_hardhat':'no_helmet','no-helmet':'no_helmet','no_helmet':'no_helmet','no helmet':'no_helmet',
    'safety-vest':'vest','safety_vest':'vest','vest':'vest','reflective-vest':'vest','hi-vis':'vest','hi_vis':'vest',
    'no-vest':'no_vest','no_vest':'no_vest','no vest':'no_vest',
    'gloves':'gloves','safety-gloves':'gloves',
    'no-gloves':'no_gloves','no_gloves':'no_gloves','no gloves':'no_gloves',
    'goggles':'goggles','safety-glasses':'goggles','glasses':'goggles','eye-protection':'goggles',
    'no-goggles':'no_goggles','no_goggles':'no_goggles','no-glasses':'no_goggles','no glasses':'no_goggles',
    'safety-boots':'safety_boots','safety_boots':'safety_boots','boots':'safety_boots',
    'no-boots':'no_safety_boots','no_boots':'no_safety_boots','no boots':'no_safety_boots',
    'person':'person','worker':'person','human':'person','people':'person',
}

def remap_class(src_id, src_names):
    if src_id >= len(src_names): return None
    name = src_names[src_id].lower().strip()
    zentra = ALIAS_MAP.get(name)
    if zentra: return CLS_MAP.get(zentra)
    for alias, target in ALIAS_MAP.items():
        if alias in name or name in alias:
            return CLS_MAP.get(target)
    return None

def read_names(ds_path):
    for yml_name in ['data.yaml','dataset.yaml']:
        y = Path(ds_path)/yml_name
        if y.exists():
            return yaml.safe_load(open(y)).get('names',[])
    for y in Path(ds_path).rglob('data.yaml'):
        return yaml.safe_load(open(y)).get('names',[])
    return []

# สร้าง merged directory
merged = Path(MERGED_DIR)
for split in ['train','val']:
    (merged/'images'/split).mkdir(parents=True, exist_ok=True)
    (merged/'labels'/split).mkdir(parents=True, exist_ok=True)

stats = {c:0 for c in ZENTRA_CLASSES}
total, skipped = 0, 0
VAL_SPLIT = 0.15

for ds_path in downloaded:
    src_names = read_names(ds_path)
    print(f'\n📁 {Path(ds_path).name} → classes: {src_names[:6]}')

    import glob
    img_files = glob.glob(f'{ds_path}/**/*.jpg', recursive=True) + \
                glob.glob(f'{ds_path}/**/*.png', recursive=True)
    random.shuffle(img_files)
    print(f'   {len(img_files)} images found')

    for img_path in img_files:
        ip = Path(img_path)
        lp = Path(str(img_path).replace('/images/','/labels/',1)).with_suffix('.txt')
        if not lp.exists():
            lp = ip.with_suffix('.txt')
        if not lp.exists():
            skipped += 1; continue

        new_lines = []
        for line in lp.read_text().strip().splitlines():
            parts = line.strip().split()
            if len(parts) < 5: continue
            new_id = remap_class(int(parts[0]), src_names)
            if new_id is None: continue
            new_lines.append(f'{new_id} {" ".join(parts[1:5])}')
            stats[ZENTRA_CLASSES[new_id]] += 1

        if not new_lines: skipped += 1; continue

        img = cv2.imread(img_path)
        if img is None: skipped += 1; continue

        split = 'val' if random.random() < VAL_SPLIT else 'train'
        stem  = f'{Path(ds_path).name}_{total:06d}'
        cv2.imwrite(str(merged/'images'/split/f'{stem}.jpg'), img, [cv2.IMWRITE_JPEG_QUALITY,92])
        (merged/'labels'/split/f'{stem}.txt').write_text('\n'.join(new_lines))
        total += 1

# เขียน dataset.yaml
yaml_path = merged/'dataset.yaml'
yaml.dump({'path':str(merged.resolve()),'train':'images/train','val':'images/val',
           'nc':len(ZENTRA_CLASSES),'names':ZENTRA_CLASSES}, open(yaml_path,'w'), allow_unicode=True)

print(f'\n✅ Merged: {total} images, {skipped} skipped')
print(f'📄 dataset.yaml → {yaml_path}')
print('\nClass distribution:')
for cls, cnt in sorted(stats.items(), key=lambda x: -x[1]):
    if cnt > 0:
        bar = '█' * min(cnt//30, 40)
        print(f'  {cls:<22} {cnt:>5} {bar}')

DATASET_YAML = str(yaml_path)

In [ ]:
# ════════════════════════════════════════════════
# CELL 7: Augmentation (เพิ่ม data 2x)
# ════════════════════════════════════════════════
import glob, time

AUG_MULTIPLIER = 2  # เพิ่ม 2 เท่า (ปรับได้ 1-4)

def augment_one(img, labels, seed):
    random.seed(seed); np.random.seed(seed)
    h, w = img.shape[:2]
    aug = img.copy()
    lines = [l.strip() for l in labels.splitlines() if l.strip()]
    new_lines = list(lines)

    # Horizontal flip
    if random.random() < 0.5:
        aug = cv2.flip(aug, 1)
        new_lines = []
        for line in lines:
            p = line.split()
            if len(p)>=5: p[1] = f'{1.0-float(p[1]):.6f}'
            new_lines.append(' '.join(p))

    # HSV jitter
    if random.random() < 0.8:
        hsv = cv2.cvtColor(aug, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[:,:,0] = np.clip(hsv[:,:,0]+random.uniform(-20,20), 0, 179)
        hsv[:,:,1] = np.clip(hsv[:,:,1]*random.uniform(0.5,1.5), 0, 255)
        hsv[:,:,2] = np.clip(hsv[:,:,2]*random.uniform(0.4,1.6), 0, 255)
        aug = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

    # Brightness/contrast
    if random.random() < 0.6:
        alpha = random.uniform(0.6, 1.4)
        beta  = random.uniform(-40, 40)
        aug = np.clip(aug.astype(np.float32)*alpha+beta, 0, 255).astype(np.uint8)

    # Blur (simulate กล้อง focus ไม่ชัด)
    if random.random() < 0.25:
        aug = cv2.GaussianBlur(aug, (random.choice([3,5]),)*2, 0)

    # Noise
    if random.random() < 0.2:
        noise = np.random.normal(0, random.uniform(5,25), aug.shape).astype(np.int16)
        aug = np.clip(aug.astype(np.int16)+noise, 0, 255).astype(np.uint8)

    # Random rotate ±10°
    if random.random() < 0.3:
        angle = random.uniform(-10, 10)
        M = cv2.getRotationMatrix2D((w/2,h/2), angle, 1.0)
        aug = cv2.warpAffine(aug, M, (w,h), borderMode=cv2.BORDER_REFLECT_101)

    return aug, '\n'.join(new_lines)

img_dir = Path(MERGED_DIR)/'images'/'train'
lbl_dir = Path(MERGED_DIR)/'labels'/'train'
orig_imgs = sorted(img_dir.glob('*.jpg'))
print(f'🎨 Augmenting {len(orig_imgs)} images × {AUG_MULTIPLIER}...')

added = 0
for i, img_path in enumerate(orig_imgs):
    img = cv2.imread(str(img_path))
    if img is None: continue
    lbl = lbl_dir/img_path.with_suffix('.txt').name
    labels = lbl.read_text() if lbl.exists() else ''
    for k in range(AUG_MULTIPLIER):
        aug_img, aug_lbl = augment_one(img, labels, i*10+k)
        stem = f'{img_path.stem}_aug{k}'
        cv2.imwrite(str(img_dir/f'{stem}.jpg'), aug_img, [cv2.IMWRITE_JPEG_QUALITY,88])
        (lbl_dir/f'{stem}.txt').write_text(aug_lbl)
        added += 1
    if i % 500 == 0: print(f'   {i}/{len(orig_imgs)}...')

total_now = len(list(img_dir.glob('*.jpg')))
print(f'\n✅ Added {added} augmented images')
print(f'   Total train set: {total_now} images')

In [ ]:
# ════════════════════════════════════════════════
# CELL 8: เริ่ม Training 🚀
# ════════════════════════════════════════════════
from ultralytics import YOLO

model = YOLO(BASE_MODEL)

print(f'\n{"═"*55}')
print(f'  🚀 ZENTRA PPE Training')
print(f'  Base model : {BASE_MODEL}')
print(f'  Dataset    : {DATASET_YAML}')
print(f'  Epochs     : {EPOCHS}')
print(f'  Batch      : {BATCH_SIZE}')
print(f'  Device     : GPU (T4)')
print(f'{"═"*55}\n')

results = model.train(
    data    = DATASET_YAML,
    epochs  = EPOCHS,
    batch   = BATCH_SIZE,
    imgsz   = IMG_SIZE,
    device  = 0,
    project = '/content/runs/ppe',
    name    = 'zentra_ppe',

    # Optimizer
    optimizer     = 'AdamW',
    lr0           = 0.001,
    lrf           = 0.01,
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 5,

    # Augmentation (YOLOv8 built-in)
    mosaic     = 1.0,
    mixup      = 0.15,
    copy_paste = 0.1,
    close_mosaic = 15,
    fliplr     = 0.5,
    hsv_h      = 0.015,
    hsv_s      = 0.7,
    hsv_v      = 0.4,
    degrees    = 5.0,
    translate  = 0.1,
    scale      = 0.6,
    shear      = 2.0,
    erasing    = 0.4,

    # Loss weights
    box = 7.5,
    cls = 0.5,
    dfl = 1.5,

    # Control
    patience    = 30,
    save        = True,
    save_period = 10,
    val         = True,
    plots       = True,
    amp         = True,   # Mixed precision → เร็วขึ้น ~40%
    workers     = 2,      # Colab: ใช้แค่ 2
    cache       = 'ram',  # Cache images ใน RAM (เร็วขึ้น)
    seed        = 42,
    verbose     = True,
)

print('\n✅ Training complete!')

In [ ]:
# ════════════════════════════════════════════════
# CELL 9: บันทึก Model ลง Google Drive
# ════════════════════════════════════════════════
import shutil, glob as _glob

# หา best.pt
candidates = _glob.glob('/content/runs/ppe/**/best.pt', recursive=True)
if not candidates:
    candidates = _glob.glob('/content/runs/**/best.pt', recursive=True)

if candidates:
    best_pt = sorted(candidates)[-1]
    print(f'✅ Found best.pt: {best_pt}')

    # บันทึกลง Google Drive
    gdrive_target = f'{GDRIVE_DIR}/ppe_finetuned.pt'
    shutil.copy2(best_pt, gdrive_target)
    print(f'💾 Saved to Drive: {gdrive_target}')

    # บันทึกใน /content ด้วย (download ง่ายกว่า)
    local_target = '/content/ppe_finetuned.pt'
    shutil.copy2(best_pt, local_target)
    print(f'📁 Local copy: {local_target}')
else:
    print('❌ ไม่พบ best.pt')

In [ ]:
# ════════════════════════════════════════════════
# CELL 10: Validate — ดูความแม่นยำ
# ════════════════════════════════════════════════
from ultralytics import YOLO

model_path = '/content/ppe_finetuned.pt'
model = YOLO(model_path)
metrics = model.val(data=DATASET_YAML, imgsz=640, verbose=True, plots=True)

print(f'\n{"═"*50}')
print(f'  📊 Results — ZENTRA PPE')
print(f'{"═"*50}')
print(f'  mAP50     : {metrics.box.map50:.4f}  ({metrics.box.map50*100:.1f}%)')
print(f'  mAP50-95  : {metrics.box.map:.4f}')
print(f'  Precision : {metrics.box.mp:.4f}')
print(f'  Recall    : {metrics.box.mr:.4f}')
print(f'{"═"*50}')

# เป้าหมาย ZENTRA: mAP50 ≥ 0.85
if metrics.box.map50 >= 0.85:
    print(f'\n🎉 ผ่านเป้าหมาย! mAP50 ≥ 85% — พร้อม deploy')
else:
    print(f'\n⚠️  ยังไม่ถึงเป้า 85% — ลอง Cell 11 (continue training)')

In [ ]:
# ════════════════════════════════════════════════
# CELL 11: Continue Training (ถ้า mAP ยังไม่พอ)
# รัน cell นี้ซ้ำได้หลายรอบ
# ════════════════════════════════════════════════
from ultralytics import YOLO

EXTRA_EPOCHS = 50   # เพิ่มกี่ epochs

model = YOLO('/content/ppe_finetuned.pt')
print(f'🔄 Continue training +{EXTRA_EPOCHS} epochs...')

results = model.train(
    data         = DATASET_YAML,
    epochs       = EXTRA_EPOCHS,
    batch        = BATCH_SIZE,
    imgsz        = IMG_SIZE,
    device       = 0,
    project      = '/content/runs/ppe',
    name         = 'zentra_ppe_continued',
    optimizer    = 'AdamW',
    lr0          = 0.0005,   # ลด lr ลง (fine-tune ต่อ)
    lrf          = 0.01,
    warmup_epochs = 2,
    mosaic       = 0.5,      # ลด mosaic ลง
    patience     = 20,
    amp          = True,
    workers      = 2,
    cache        = 'ram',
    verbose      = True,
)

# Update model
import glob as _g, shutil
new_best = sorted(_g.glob('/content/runs/ppe/zentra_ppe_continued*/best.pt'))[-1]
shutil.copy2(new_best, '/content/ppe_finetuned.pt')
shutil.copy2(new_best, f'{GDRIVE_DIR}/ppe_finetuned.pt')
print(f'\n✅ Updated model saved')

In [ ]:
# ════════════════════════════════════════════════
# CELL 12: Export ONNX (สำหรับ deploy ใน ZENTRA)
# ════════════════════════════════════════════════
from ultralytics import YOLO

model = YOLO('/content/ppe_finetuned.pt')
out = model.export(format='onnx', imgsz=640, simplify=True, opset=17)
print(f'✅ ONNX exported: {out}')

import shutil
shutil.copy2(out, f'{GDRIVE_DIR}/ppe_finetuned.onnx')
print(f'💾 Saved to Drive: {GDRIVE_DIR}/ppe_finetuned.onnx')

In [ ]:
# ════════════════════════════════════════════════
# CELL 13: Download ไฟล์ลงเครื่อง
# ════════════════════════════════════════════════
from google.colab import files

print('📥 Downloading ppe_finetuned.pt...')
files.download('/content/ppe_finetuned.pt')

# ดู training curves
import glob as _g
result_imgs = _g.glob('/content/runs/ppe/**/results.png', recursive=True)
if result_imgs:
    from IPython.display import Image, display
    display(Image(result_imgs[-1]))
    print('📊 Training curves แสดงด้านบน')
    files.download(result_imgs[-1])

print('\n✅ เสร็จสิ้น!')
print('   นำ ppe_finetuned.pt ไปใส่ที่: zentra/models/ppe_finetuned.pt')
print('   แล้วแก้ config.py:')
print('     USE_LOCAL_MODEL = True')
print('     PPE_LOCAL_MODEL = "models/ppe_finetuned.pt"')

---
## หลัง Download ไฟล์แล้ว → ใส่ใน ZENTRA

```
zentra/
└── models/
    └── ppe_finetuned.pt   ← วางตรงนี้
```

แก้ `config.py`:
```python
USE_LOCAL_MODEL = True
PPE_LOCAL_MODEL = 'models/ppe_finetuned.pt'
```

---
## เป้าหมายความแม่นยำ

| Metric | เป้าหมาย ZENTRA |
|--------|----------------|
| mAP50  | ≥ 85% ✅        |
| mAP50-95 | ≥ 60%        |
| Precision | ≥ 80%      |
| Recall | ≥ 80%          |

ถ้าไม่ถึง → รัน **Cell 11** ซ้ำได้เลย